In [1]:
%pip install -qU langchain-community pymupdf
!pip install -qU langchain-huggingface sentence-transformers
!pip install -qU langchain-groq
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 116.4 MB/s eta 0:00:00


# 1. Loading the document

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "/content/the-quran-with-annotated-interpretation-in-modern-english-ali-unal.pdf"
loader = PyMuPDFLoader(file_path)

In [3]:
docs = loader.load()
# skiping empty pages
non_empty_docs = [d for d in docs if d.page_content.strip()]

# 2. Spliting document into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000, # 10000 charecters long text
    chunk_overlap=200, # 200 charecters long overlapping
)

split_docs = text_splitter.split_documents(non_empty_docs)

# 3. Embeddings Model

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
texts = [doc.page_content for doc in split_docs] # convert documents into list[str]

In [7]:
split_docs_embeddings = embed_model.embed_documents(texts) # generate embeddings (list[list[float]])

# 4. FAISS (Facebook AI Similarity Search) vector database

In [8]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(
    documents=split_docs,
    embedding=embed_model,
)

# 5. LLM. GROQ (llama-3.3-70b-versatile)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="your_api_key"
)

# 6. Building Prompts and chains

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [11]:
def ask(query):
    top_docs = faiss_db.similarity_search(query, k=10)
    prompt_template = PromptTemplate.from_template(
      "Give answer according to the following passages in the quran {context}"
      "If the answer is not present in the given context then give answer according to the internet sources."
      "But do inform that there are no passages in the quran about the question."
      "Answer the following question {question}."
    )
    chain = prompt_template | llm | StrOutputParser()
    result = chain.invoke({"context": top_docs, "question": query})
    return result

Relevant questions

In [12]:
asnwer = ask("What does the Quran say about Day of Judgment?")
print(asnwer)

The Quran mentions the Day of Judgment in several passages. According to the Quran, the Day of Judgment is the day when all souls will be judged by God based on their deeds in this life. 

In Surah 82, verses 17-19, the Quran describes the Day of Judgment as a day when no soul will have the power to do anything in favor of another, and the command will be God's entirely and exclusively. 

In Surah 21, verse 47, the Quran states that God will set up balances of absolute justice on the Day of Resurrection, and no person will be wronged in the least. Every deed, no matter how small, will be weighed and accounted for.

In Surah 6, verse 60, the Quran mentions that God recalls the souls of the dead and knows what they have done in their lifetimes. On the Day of Judgment, God will raise the souls again and make them understand what they were doing.

In Surah 25, verse 2, the Quran emphasizes God's power and knowledge, stating that He is the one who will judge on the Day of Judgment.

In Sura

In [13]:
asnwer = ask("What of someone donot FAST in the month of RAMADAN?")
print(asnwer)

According to the Quran and Islamic teachings, fasting during the month of Ramadan is one of the Five Pillars of Islam and is obligatory for all adult Muslims who are physically and mentally able. 

If someone does not fast during Ramadan without a valid reason, it is considered a serious sin. The Quran states: "And for those who cannot afford it, fasting is compulsory" (Surah Al-Baqarah, 2:183-184).

However, there are some exceptions and circumstances where a person may be exempt from fasting, such as:

1. Illness or sickness: If a person is suffering from a serious illness or sickness that makes it difficult or harmful for them to fast, they are exempt from fasting.
2. Travel: If a person is traveling during Ramadan, they are allowed to break their fast, but they must make up the missed days later.
3. Menstruation: Women who are menstruating or experiencing post-childbirth bleeding are exempt from fasting.
4. Pregnancy or breastfeeding: Pregnant or breastfeeding women who are concern

Irrelevant questions

In [14]:
asnwer = ask("When did dinosaurs came into being?")
print(asnwer)

There is no passage in the Quran that specifically mentions dinosaurs or their existence. The Quran does mention the creation of animals and the natural world, but it does not provide a detailed account of the history of life on Earth or the evolution of specific species.

According to internet sources, dinosaurs are believed to have originated during the Middle to Late Triassic period, around 230-245 million years ago. They dominated Earth's landscapes during the Mesozoic Era, which lasted from about 252 million to 66 million years ago. The most well-known dinosaurs, such as Tyrannosaurus rex, Velociraptor, and Diplodocus, lived during the Jurassic and Cretaceous periods.

It's worth noting that the Quran is a religious text that focuses on spiritual and moral guidance, rather than providing a scientific or historical account of the natural world. While the Quran does contain references to the natural world and the creation of living things, it does not provide a detailed or scientifi